# alpha-forecast — demo

A quick tour of the forecasting & backtesting workflow:
**data → causal features → multi-model walk-forward backtest → honest evaluation.**

Runs offline on synthetic data; set a real ticker (and install `yfinance`) for live data.

In [ ]:
import numpy as np
import pandas as pd

from alpha_forecast.data import load_prices
from alpha_forecast.features import build_features, load_fama_french
from alpha_forecast.pipeline import run_comparison

SYMBOL = "AAPL"  # falls back to synthetic data offline

## 1. Load prices

In [ ]:
prices = load_prices(SYMBOL, start="2018-01-01", end="2023-12-31")
prices.tail()

## 2. Build causal features (optionally with Fama-French factors)

In [ ]:
ff = load_fama_french("2018-01-01", "2023-12-31")
X, y = build_features(prices, horizon=1, extra=ff)
print(X.shape)
X.head()

## 3. Compare models with walk-forward backtesting

A model is only interesting if it beats the **naive** baseline.

In [ ]:
board = run_comparison(SYMBOL, start="2018-01-01", end="2023-12-31", horizon=1, use_factors=True)
board

## 4. Plot the best model's equity curve

In [ ]:
from alpha_forecast.backtest import walk_forward_backtest
from alpha_forecast.evaluation.plots import equity_curve
from alpha_forecast.models import MODEL_REGISTRY

best = board.iloc[0]['model']
res = walk_forward_backtest(MODEL_REGISTRY[best], X, y)
equity_curve(res.actuals, res.predictions)

## 5. Forecast volatility (GARCH)

In [ ]:
from alpha_forecast.models.volatility import rolling_vol_forecast

rets = np.log(prices['close']).diff().dropna()
vol = rolling_vol_forecast(rets)
vol.tail()